# v27 Standard — TRUE freeze MC + YNU-only grid

MC is copied directly from v25 preds. If YNU grid does not beat v25, selected output rolls back to v25.

In [ ]:
import os, re, json, glob
from pathlib import Path
from collections import Counter

LABELS = ["A","B","C","D","Yes","No","Unknown"]
MC_LABELS = {"A","B","C","D"}
YNU_LABELS = {"Yes","No","Unknown"}
OUT_DIR = Path("/kaggle/working/v27_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def norm_answer(x):
    if x is None: return "UNPARSEABLE"
    s = str(x).strip()
    if not s: return "UNPARSEABLE"
    low = s.lower()
    if low in ["a","option a","answer a"]: return "A"
    if low in ["b","option b","answer b"]: return "B"
    if low in ["c","option c","answer c"]: return "C"
    if low in ["d","option d","answer d"]: return "D"
    if low in ["yes","true","entailed","supported"]: return "Yes"
    if low in ["no","false","not entailed","unsupported","contradicted"]: return "No"
    if low in ["unknown","uncertain","not enough information","cannot determine"]: return "Unknown"
    m = re.search(r"\b(final answer|answer)\s*[:\-]\s*(A|B|C|D|Yes|No|Unknown)\b", s, flags=re.I)
    if m: return norm_answer(m.group(2))
    return s if s in LABELS else "UNPARSEABLE"

def to_jsonable(x):
    try:
        import numpy as np
        if isinstance(x, (np.integer,)): return int(x)
        if isinstance(x, (np.floating,)): return float(x)
        if isinstance(x, (np.bool_,)): return bool(x)
    except Exception:
        pass
    if isinstance(x, dict): return {str(k): to_jsonable(v) for k,v in x.items()}
    if isinstance(x, (list, tuple)): return [to_jsonable(v) for v in x]
    return x

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(to_jsonable(obj), f, ensure_ascii=False, indent=2)
    print("Saved:", path)

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def resolve_one(patterns, required=True, label="file"):
    hits = []
    for pat in patterns:
        hits += glob.glob(pat, recursive=True)
    hits = sorted(set(hits))
    print(f"{label} candidates:", hits[:20], "..." if len(hits) > 20 else "")
    if not hits:
        if required: raise FileNotFoundError(f"Missing {label}; checked {patterns}")
        return None
    hits = sorted(hits, key=lambda p: (0 if "/kaggle/input/" in p else 1, 0 if "v25_operational_preds" in Path(p).name else 1, len(p)))
    print("Using", label, ":", hits[0])
    return Path(hits[0])

def compute_metrics(rows, name="metric"):
    y_true = [norm_answer(r.get("gold")) for r in rows]
    y_pred = [norm_answer(r.get("pred")) for r in rows]
    n = len(rows)
    acc = sum(t == p for t,p in zip(y_true,y_pred)) / max(1,n)
    per, f1s, weights = {}, [], []
    for lab in LABELS:
        tp = sum(t == lab and p == lab for t,p in zip(y_true,y_pred))
        fp = sum(t != lab and p == lab for t,p in zip(y_true,y_pred))
        fn = sum(t == lab and p != lab for t,p in zip(y_true,y_pred))
        support = sum(t == lab for t in y_true)
        pred_count = sum(p == lab for p in y_pred)
        precision = tp/(tp+fp) if tp+fp else 0.0
        recall = tp/(tp+fn) if tp+fn else 0.0
        f1 = 2*precision*recall/(precision+recall) if precision+recall else 0.0
        per[lab] = dict(precision=precision, recall=recall, f1=f1, support=support, pred_count=pred_count, tp=tp, fp=fp, fn=fn)
        f1s.append(f1); weights.append(support)
    return dict(
        name=name, n=n, acc=acc, macro_f1=sum(f1s)/len(f1s),
        weighted_f1=sum(f*w for f,w in zip(f1s,weights))/max(1,sum(weights)),
        mc_macro_f1=sum(per[x]["f1"] for x in ["A","B","C","D"])/4,
        ynu_macro_f1=sum(per[x]["f1"] for x in ["Yes","No","Unknown"])/3,
        per_label=per,
    )

def print_metric(m):
    print("="*100)
    print(m["name"])
    print("="*100)
    print(f"N={m['n']} acc={m['acc']:.4f} macro={m['macro_f1']:.4f} weighted={m['weighted_f1']:.4f}")
    print(f"MC={m['mc_macro_f1']:.4f} YNU={m['ynu_macro_f1']:.4f}")
    print("-"*100)
    for lab in LABELS:
        d = m["per_label"][lab]
        print(f"{lab:<8} P={d['precision']:.3f} R={d['recall']:.3f} F1={d['f1']:.3f} gold={d['support']} pred={d['pred_count']}")

def load_v25_preds():
    p = resolve_one([
        "/kaggle/input/**/v25_operational_preds*.json",
        "/kaggle/input/**/v25*preds*.json",
        "/kaggle/working/**/v25_operational_preds*.json",
        "/kaggle/working/**/v25*preds*.json",
    ], required=True, label="v25 operational preds")
    rows = load_json(p)
    print("Loaded v25 rows:", len(rows))
    return rows, p

def load_candidates(required=False):
    p = resolve_one([
        "/kaggle/input/**/v23_val_candidates.json",
        "/kaggle/input/**/v23*candidates*.json",
        "/kaggle/working/**/v23_val_candidates.json",
        "/kaggle/working/**/v23*candidates*.json",
    ], required=required, label="v23 candidates")
    if p is None: return None, None
    rows = load_json(p)
    print("Loaded candidate rows:", len(rows))
    return rows, p

def assert_mc_frozen(base, rows):
    bad = []
    for b,r in zip(base, rows):
        if bool(b.get("is_mc")) and norm_answer(b.get("pred")) != norm_answer(r.get("pred")):
            bad.append((b.get("idx"), b.get("pred"), r.get("pred")))
    if bad:
        print("MC freeze violations:", bad[:20])
        raise AssertionError(f"MC freeze violated in {len(bad)} rows")
    print("MC freeze check: PASS")

def cand_map(candidates):
    out = {}
    for i,r in enumerate(candidates or []):
        idx = r.get("idx", r.get("id", r.get("record_idx", i)))
        out[int(idx)] = r
    return out

def c_answer(c):
    return norm_answer(c.get("answer") or c.get("pred") or c.get("final_answer"))

def get_cands(row, cmap):
    return (cmap.get(int(row.get("idx", -1))) or {}).get("candidates", [])

def score_cand(c, p):
    ans = c_answer(c)
    supp = c.get("supporting_premises") or c.get("supporting") or []
    try: supp_len = len(supp)
    except Exception: supp_len = 0
    txt = str(c.get("text") or c.get("raw_text") or c.get("explanation") or "")
    score = 0.0
    score += p.get("w_vote", 1.0) * float(c.get("vote_share", 0.0) or 0.0)
    score += p.get("w_cite", 0.0) * (1.0 if supp_len > 0 else 0.0)
    score += p.get("w_short_supp", 0.0) * (1.0 if 1 <= supp_len <= 5 else 0.0)
    if supp_len > 8: score -= p.get("p_long_supp", 0.0)
    if ans == "Yes": score += p.get("yes_boost", 0.0)
    if ans == "No": score -= p.get("no_penalty", 0.0)
    if ans == "Unknown": score -= p.get("unk_penalty", 0.0)
    if re.search(r"\b(however|contradict|cannot conclude|not enough|unknown)\b", txt, flags=re.I):
        score -= p.get("p_contra_text", 0.0)
    return score

def choose_ynu(row, cmap, params):
    cs = [c for c in get_cands(row, cmap) if c_answer(c) in YNU_LABELS]
    if not cs: return norm_answer(row.get("pred"))
    return c_answer(max(cs, key=lambda c: score_cand(c, params)))

def build_rows(base, cmap=None, params=None, mode="baseline"):
    rows = []
    for r in base:
        rr = dict(r)
        if bool(r.get("is_mc")):
            rr["pred"] = norm_answer(r.get("pred"))
            rr["source"] = "mc_frozen_from_v25"
        else:
            if mode == "candidate_rerank" and cmap is not None:
                rr["pred"] = choose_ynu(r, cmap, params or {})
                rr["source"] = "ynu_candidate_rerank_v27"
            elif mode == "oracle_ynu" and cmap is not None:
                gold = norm_answer(r.get("gold"))
                answers = [c_answer(c) for c in get_cands(r, cmap)]
                rr["pred"] = gold if gold in answers else norm_answer(r.get("pred"))
                rr["source"] = "ynu_oracle_analysis"
            else:
                rr["pred"] = norm_answer(r.get("pred"))
                rr["source"] = "ynu_frozen_from_v25"
        rows.append(rr)
    assert_mc_frozen(base, rows)
    return rows

In [ ]:
base_rows, base_path = load_v25_preds()
base_metric = compute_metrics(base_rows, "v25_baseline_exact")
print_metric(base_metric)

candidates, cand_path = load_candidates(required=True)
cmap = cand_map(candidates)
print("Candidate map size:", len(cmap))

In [ ]:
grid = []
for w_vote in [0.0, 0.5, 0.8, 1.0, 1.2]:
  for w_cite in [0.0, 0.2, 0.4]:
    for w_short in [0.0, 0.1, 0.2]:
      for yes_boost in [0.0, 0.2, 0.4, 0.6]:
        for no_penalty in [0.0, 0.15, 0.3]:
          for unk_penalty in [0.0, 0.15, 0.3, 0.5]:
            params = dict(w_vote=w_vote, w_cite=w_cite, w_short_supp=w_short, yes_boost=yes_boost,
                          no_penalty=no_penalty, unk_penalty=unk_penalty, p_long_supp=0.2, p_contra_text=0.4)
            rows = build_rows(base_rows, cmap, params, mode="candidate_rerank")
            m = compute_metrics(rows, "attempt")
            obj = m["macro_f1"] + 0.25*m["ynu_macro_f1"] + 0.15*m["per_label"]["Yes"]["f1"] + 0.10*m["per_label"]["Unknown"]["f1"]
            grid.append(dict(obj=obj, **params, acc=m["acc"], macro_f1=m["macro_f1"], mc_macro_f1=m["mc_macro_f1"],
                             ynu_macro_f1=m["ynu_macro_f1"], yes_f1=m["per_label"]["Yes"]["f1"],
                             no_f1=m["per_label"]["No"]["f1"], unknown_f1=m["per_label"]["Unknown"]["f1"]))
grid = sorted(grid, key=lambda x: x["obj"], reverse=True)
print("Top grid rows:")
for r in grid[:20]: print(r)

best_params = {k:grid[0][k] for k in ["w_vote","w_cite","w_short_supp","yes_boost","no_penalty","unk_penalty","p_long_supp","p_contra_text"]}
best_rows = build_rows(base_rows, cmap, best_params, mode="candidate_rerank")
best_metric = compute_metrics(best_rows, "v27_standard_best_attempt")
print_metric(best_metric)

In [ ]:
if best_metric["macro_f1"] > base_metric["macro_f1"] + 1e-12:
    selected_rows = best_rows
    selected_source = "rerank_beats_v25"
else:
    selected_rows = build_rows(base_rows, None, None, mode="baseline")
    selected_source = "rollback_to_v25"
selected_metric = compute_metrics(selected_rows, "v27_standard_SELECTED")
print("Selected:", selected_source)
print_metric(selected_metric)

summary = dict(version="v27_standard_true_freeze_mc", base_path=str(base_path), candidate_path=str(cand_path),
               base_metric=base_metric, best_params=best_params, best_attempt_metric=best_metric,
               selected_source=selected_source, selected_metric=selected_metric,
               mc_freeze="strict_copy_from_v25_preds")
save_json(summary, OUT_DIR / "v27_standard_summary.json")
save_json(selected_rows, OUT_DIR / "v27_standard_preds.json")
save_json(grid[:200], OUT_DIR / "v27_standard_grid_rows.json")